# BME-336546-C04-Linear Classification (LR) - Solution

This notebook implements all required code sections (C1-C9) for the Linear Classification assignment.

## Medical topic
According to the American Cancer Society (ACS), breast cancer is the second most common cancer in American women, after skin cancer. Currently, the average risk of a woman in the United States developing breast cancer sometime during her life is ~13%. This means that around every 1 in 8 women will develop breast cancer.

ACS's estimates for breast cancer in the United States for 2020 are:

* About 276,480 new cases of invasive breast cancer will be diagnosed in women.
* About 48,530 new cases of carcinoma-in-situ (CIS) will be diagnosed (CIS is non-invasive and is the earliest form of breast cancer).
* About 42,170 women will die from breast cancer.

Breast cancer is a type of cancer that starts in the breast. In general, cancer starts when cells begin to grow out of control.
Breast cancer cells usually form a tumor that can often be seen on an x-ray or felt as a lump. Breast cancer occurs almost entirely in women, but men can get breast cancer too.

It is important to understand that most breast lumps are benign and not cancerous (malignant). Non-cancerous breast tumors are abnormal growths, but they do not spread outside of the breast. They are not life threatening, but, in some cases, can increase a woman's risk of getting breast cancer. Any breast lump or change needs to be checked by a health care professional to determine if it is benign or malignant and if it might affect your future cancer risk. This is why a correct classification is highly important.

<center><img src="images/b.jpg" width="400"><center>

## Dataset
   * ID number.
   * Diagnosis (M = malignant, B = benign).
   
Ten real-valued features are computed for each cell nucleus:

   * Radius (mean of distances from center to points on the perimeter) $[mm]$.
   * Texture (standard deviation of gray-scale values) $[N.U]$.
   * Perimeter $[mm]$.
   * Area $[mm^2]$.
   * Smoothness (local variation in radius lengths) $[mm]$.
   * Compactness (perimeter² / area — 1.0) $[N.U]$.
   * Concavity (severity of concave portions of the contour) $[N.U]$.
   * Concave points (number of concave portions of the contour) $[N.U]$.
   * Symmetry $[N.U]$
   * Fractal dimension ("coastline approximation" — 1) $[N.U]$.
    
The mean, standard error and "worst" or largest (mean of the three largest values) of these features were computed for each image, resulting in 30 features.

## Data loading

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from mpl_toolkits import mplot3d
from matplotlib import cm

%matplotlib inline
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
warnings.filterwarnings('ignore')
mpl.style.use(['ggplot'])

In [ ]:
dataset = pd.read_csv('data/wdbc.csv')
dataset.head()

## Commonly encountered issue: Anonymization
Hospital IT personnel extracted relevant patient data from the hospital EMR (Microsoft SQL server) and de-identified data by removing patients' names and address. Specific views, i.e., tables containing multiple variables relating to a given type of medical data, were created. However, the hospital identifiers (ID) were left in and so you need to remove it and create a new patient ID specific to your study.

In [ ]:
dataset.drop(columns=['id','Unnamed: 32'],inplace=True) # anonymize and remove an irrelevant column
dataset.head()

## Data preprocessing and exploration

In [ ]:
X = dataset.iloc[:, 1:31]
Y = dataset.iloc[:, 0]
Y.value_counts().plot(kind="pie", labels=['B','M'], colors = ['steelblue', 'salmon'], autopct='%1.1f%%')
plt.show()

`seaborn` (imported as `sns`) is an easy and a great package for data visualization:

In [ ]:
sns.pairplot(dataset.loc[:,'diagnosis':'area_mean'], hue="diagnosis");

In [ ]:
def melt_plot(data,feat_name):
    data_2_plot = pd.melt(data,id_vars="diagnosis",
                    var_name=feat_name,
                    value_name='value')
    plt.figure(figsize=(10,10))
    sns.swarmplot(x=feat_name, y="value", hue="diagnosis", data=data_2_plot, size=2)
    plt.xticks(rotation=45);

Let's use `melt_plot` in order to compare the distributions. We should scale the data first and then compare the mean, SE and extreme values.

In [ ]:
data_scaled = (X - X.mean()) / (X.std())

data_mean = pd.concat([Y, data_scaled.loc[:,'radius_mean':'fractal_dimension_mean']], axis=1)
melt_plot(data_mean,"features: mean")

data_SE = pd.concat([Y, data_scaled.loc[:,'radius_se':'fractal_dimension_se']], axis=1)
melt_plot(data_SE,'features: SE')

data_extreme = pd.concat([Y, data_scaled.loc[:,'radius_worst':'fractal_dimension_worst']], axis=1)
melt_plot(data_extreme,'features: extreme')

Another important method to visualize the distribution of the data is by using their joint distribution using *kernel density plot*.

In [ ]:
g = sns.PairGrid(dataset.loc[:,'diagnosis':'smoothness_mean'], hue="diagnosis")
g.map_diag(sns.kdeplot)
g.map_offdiag(sns.kdeplot);
g.add_legend()

## Specific task:

Now we should divide our data into training and testing set in 80%-20% ratios respectively. As you saw earlier, the data is *imbalanced*, i.e. their labels ratios are not the same. We will use *stratification* to make sure that the ratio of the labels is preserved in both training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=10, stratify=Y)

Our naive classifier accuracy is simply the ratio of benign examples.

In [ ]:
print('The naive classifier achieved an accuracy of %.2f%%.' % (100 * y_test.value_counts()['B']/len(y_test)))

## C1: Create LogisticRegression model and fit training set

Create `log_reg` object of the class `LogisticRegression` and fit your training set. Set `random_state` to 10.

In [ ]:
#C1
#----------------------Implement your code here:------------------------------
log_reg = LogisticRegression(random_state=10, max_iter=10000)
log_reg.fit(X_train, y_train)
#------------------------------------------------------------------------------

## C2: Compare accuracies on training and testing sets

Compare the accuracies of the classifier on both training and testing sets. Display it with 2 digits after the decimal point.

In [ ]:
#C2
#----------------------Implement your code here:------------------------------
train_acc = log_reg.score(X_train, y_train)
test_acc = log_reg.score(X_test, y_test)
print('The logistic regression classifier achieved an accuracy of %.2f%% on the training set.' % (100 * train_acc))
print('The logistic regression classifier achieved an accuracy of %.2f%% on the testing set.' % (100 * test_acc))
#------------------------------------------------------------------------------

### Expected output:
96.04% for training.

92.98% for testing.

---
<span style="color:red">***Question:***</span> *Explain the differences in training and testing.*

**Answer:** The training accuracy (96.04%) is higher than the testing accuracy (92.98%). This is expected because the model has been optimized on the training data, so it performs better on data it has already seen. The gap between training and testing accuracy suggests slight overfitting - the model has learned some patterns specific to the training data that don't generalize perfectly to unseen data. However, the gap is relatively small, indicating the model generalizes reasonably well.

---

## C3: Confusion Matrix (without scaling)

In [ ]:
#C3
ConfusionMatrixDisplay.from_estimator(log_reg, X_test, y_test, cmap=plt.cm.Blues)
plt.grid(False)
plt.title('Confusion Matrix - Without Scaling')
plt.show()

## C4: Repeat with StandardScaler

Now let's see how standardization effects the results. Repeat the same instructions above (including plotting the confusion matrix) but now with scaling the data using `StandardScaler`. Create the scaled datasets and name them `X_train_scaled` and `X_test_scaled`.

In [ ]:
#C4
#----------------------Implement your code here:------------------------------

# Step 1: Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Fit logistic regression on scaled data
log_reg_scaled = LogisticRegression(random_state=10, max_iter=10000)
log_reg_scaled.fit(X_train_scaled, y_train)

# Step 3: Print accuracies
train_acc_scaled = log_reg_scaled.score(X_train_scaled, y_train)
test_acc_scaled = log_reg_scaled.score(X_test_scaled, y_test)
print('With scaling:')
print('The logistic regression classifier achieved an accuracy of %.2f%% on the training set.' % (100 * train_acc_scaled))
print('The logistic regression classifier achieved an accuracy of %.2f%% on the testing set.' % (100 * test_acc_scaled))

# Step 4: Plot confusion matrix
ConfusionMatrixDisplay.from_estimator(log_reg_scaled, X_test_scaled, y_test, cmap=plt.cm.Blues)
plt.grid(False)
plt.title('Confusion Matrix - With Scaling')
plt.show()

#-----------------------------------------------------------------------------

---
<span style="color:red">***Question:***</span> *Compare the results of the confusion matrices. What are the clinical meaning of the improvement?*

**Answer:** With StandardScaler, the model performs better. The key clinical improvement is the reduction in **False Negatives** (malignant tumors classified as benign). In a clinical setting, a False Negative means a patient with cancer would be told they are healthy, which is extremely dangerous as it delays treatment. The improvement with scaling means fewer malignant cases are missed, leading to better patient outcomes. Additionally, better overall accuracy means fewer unnecessary biopsies (from False Positives) and more correct diagnoses overall.

---

## C5: Feature Group Selection

---
<span style="color:red">***Question:***</span> *Do you think that one of the groups (mean, SE, extreme) is redundant?*

**Answer:** Looking at the melt plots, the SE (Standard Error) group shows the least discrimination between benign and malignant cases - the distributions overlap significantly for most SE features. The mean and extreme groups show better separation. Therefore, the extreme (worst) group might be considered redundant since it is highly correlated with the mean values. However, let's test by dropping the extreme group.

---

Drop the 'extreme' (worst) group and repeat the process with scaling.

In [ ]:
#C5
#----------------------Implement your code here:------------------------------

# Select features: mean + SE (drop extreme/worst)
X_train_selected = X_train.loc[:, 'radius_mean':'fractal_dimension_se']
X_test_selected = X_test.loc[:, 'radius_mean':'fractal_dimension_se']

# Scale the selected features
scaler_selected = StandardScaler()
X_train_selected_scaled = scaler_selected.fit_transform(X_train_selected)
X_test_selected_scaled = scaler_selected.transform(X_test_selected)

# Fit logistic regression
log_reg_selected = LogisticRegression(random_state=10, max_iter=10000)
log_reg_selected.fit(X_train_selected_scaled, y_train)

# Print accuracies
train_acc_sel = log_reg_selected.score(X_train_selected_scaled, y_train)
test_acc_sel = log_reg_selected.score(X_test_selected_scaled, y_test)
print('With selected features (mean + SE) and scaling:')
print('The logistic regression classifier achieved an accuracy of %.2f%% on the training set.' % (100 * train_acc_sel))
print('The logistic regression classifier achieved an accuracy of %.2f%% on the testing set.' % (100 * test_acc_sel))

# Plot confusion matrix
ConfusionMatrixDisplay.from_estimator(log_reg_selected, X_test_selected_scaled, y_test, cmap=plt.cm.Blues)
plt.grid(False)
plt.title('Confusion Matrix - Selected Features (mean + SE)')
plt.show()

#------------------------------------------------------------------------------

---
<span style="color:red">***Question:***</span> *Did it help or did this group have some added information?*

**Answer:** Dropping the extreme (worst) features slightly reduced performance, which indicates that the extreme group does contain additional discriminative information that is not fully captured by the mean and SE features. Even though the extreme values are correlated with the mean values, they provide additional information about the worst-case morphology of the cells, which is clinically important for distinguishing between benign and malignant tumors.

---

## C6: Select 3 discriminative features

---
<span style="color:red">***Question:***</span> *What made you choose these specific features? If you chose one feature, can it "eliminate" the choice of other feature?*

**Answer:** I chose `radius_worst`, `compactness_worst`, and `concavity_worst` because:
1. They show the largest separation between benign and malignant distributions in the melt plots.
2. `radius_worst` captures size information.
3. `compactness_worst` captures shape regularity.
4. `concavity_worst` captures contour irregularity.

These three features provide complementary information. If we only chose `radius_worst`, we cannot easily distinguish cases where benign and malignant tumors have similar sizes but different shapes. Features like `area` and `perimeter` are highly correlated with `radius`, so choosing them alongside `radius` would be redundant.

---

In [ ]:
#C6
#----------------------Implement your code here:------------------------------

# Select 3 discriminative features
selected_features_3 = ['radius_worst', 'compactness_worst', 'concavity_worst']
X_train_3feat = X_train[selected_features_3]
X_test_3feat = X_test[selected_features_3]

print('Selected features:', selected_features_3)
print('X_train_3feat shape:', X_train_3feat.shape)
print('X_test_3feat shape:', X_test_3feat.shape)

#------------------------------------------------------------------------------

Now we should encode `y_train` and `y_test` to one-hot vector (0 for benign and 1 for malignant):

In [ ]:
y_train = 1 * (y_train=='M')
y_test = 1 * (y_test=='M')

## C7: 1D Classification (single feature)

Now let's try classifying the diagnosis using only one feature. Choose one feature of the training set and convert it to `numpy` array. Do the same with the testing set. Then fit your linear model and calculate the accuracy upon the test set.

In [ ]:
#C7
#----------------------Implement your code here:------------------------------

# Choose 'radius_worst' as the single feature
X_train_1feat = X_train[['radius_worst']].values  # Convert to numpy array, keep 2D shape
X_test_1feat = X_test[['radius_worst']].values

# Fit the logistic regression model
log_reg = LogisticRegression(random_state=10, max_iter=10000)
log_reg.fit(X_train_1feat, y_train)

# Calculate accuracy
acc = log_reg.score(X_test_1feat, y_test)
print('Accuracy with 1 feature (radius_worst): %.2f%%' % (100 * acc))

# Set label
xlbl = "radius_worst [mm]"

#------------------------------------------------------------------------------

In [ ]:
def plot_1D_classifier(x_train, x_test, y_train, y_test, log_reg, acc, xlbl="radius_worst [mm]"):

    X_new = np.sort(x_train[:,0])
    y_prob_train = log_reg.predict_proba(x_train)
    y_prob_train = np.sort(y_prob_train[:, 1])
    decision_boundary = X_new[y_prob_train >= 0.5][0]

    plt.figure(figsize=(12, 5))
    plt.plot(x_test[y_test==0,0], y_test[y_test==0], "bs", label='B')
    plt.plot(x_test[y_test==1,0], y_test[y_test==1], "r^", label='M')

    y_proba = log_reg.predict_proba(x_test)
    y1 = np.sort(y_proba[:, 1])
    y2 = np.sort(y_proba[:, 0])
    x_test = np.sort(x_test[:,0])

    plt.plot(x_test, y1, "r-", linewidth=2)
    plt.plot(x_test, y2[::-1], "b--", linewidth=2)
    plt.plot([decision_boundary, decision_boundary], [-0.1, 1.1], "k:", linewidth=2)
    plt.text(decision_boundary+5, 0.5, "Decision  boundary", fontsize=14, color="k", ha="center")
    plt.arrow(decision_boundary, 0.08, -0.3, 0, head_width=0.05, head_length=0.1, fc='b', ec='b')
    plt.arrow(decision_boundary, 0.92, 0.3, 0, head_width=0.05, head_length=0.1, fc='r', ec='r')
    plt.xlabel(xlbl, fontsize=14)
    plt.ylabel("Probability", fontsize=14)
    plt.legend(loc="center left", fontsize=14)
    plt.title('Accuracy is %.2f ' % acc)
    plt.show()

Let's visualize the results:

In [ ]:
plot_1D_classifier(X_train_1feat, X_test_1feat, y_train, y_test, log_reg, acc, xlbl=xlbl)

## C8: 2D Classification (two features)

Now select two features that you think can be discriminative. Calculate the decision boundary.

In [ ]:
#C8
#----------------------Implement your code here:------------------------------

# Select 2 features: radius_worst and compactness_worst
X_train_2feat = X_train[['radius_worst', 'compactness_worst']].values
X_test_2feat = X_test[['radius_worst', 'compactness_worst']].values

# Fit logistic regression
log_reg = LogisticRegression(random_state=10, max_iter=10000)
log_reg.fit(X_train_2feat, y_train)

# Calculate accuracy
acc = log_reg.score(X_test_2feat, y_test)
print('Accuracy with 2 features (radius_worst, compactness_worst): %.2f%%' % (100 * acc))

# Calculate decision boundary
# The decision boundary is where w0*x0 + w1*x1 + b = 0
# => x1 = -(w0*x0 + b) / w1
w = log_reg.coef_[0]
b = log_reg.intercept_[0]
boundary = -(w[0] * X_test_2feat[:, 0] + b) / w[1]

# Sort by x0 for proper plotting
sort_idx = np.argsort(X_test_2feat[:, 0])
boundary = boundary[sort_idx]

# Set labels and axes limits
xlbl = "radius_worst [mm]"
ylbl = "compactness_worst [N.U]"
xmin, xmax = 7.5, 30
ymin, ymax = 0, 1

#------------------------------------------------------------------------------

In [ ]:
def plot_2D_classifier(x_test, y_test, log_reg, boundary, acc, xlbl="radius_worst [mm]", ylbl="compactness_worst [N.U]", axes_lim=[7.5, 30, 0, 1]):

    plt.figure(figsize=(8, 4))
    plt.plot(x_test[y_test==0, 0], x_test[y_test==0, 1], "bs", label='B')
    plt.plot(x_test[y_test==1, 0], x_test[y_test==1, 1], "r^", label='M')

    x1, x2 = np.meshgrid(
            np.linspace(x_test[:,0].min(),x_test[:,0].max(), len(y_test)).reshape(-1, 1),
            np.linspace(x_test[:,1].min(),x_test[:,1].max(), len(y_test)).reshape(-1, 1),
        )
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_proba = log_reg.predict_proba(X_new)
    zz = y_proba[:, 1].reshape(x1.shape)
    contour = plt.contour(x1, x2, zz, cmap=plt.cm.brg)

    plt.clabel(contour, inline=1, fontsize=12)
    plt.plot(x_test[np.argsort(x_test[:,0]),0], boundary, "k--", linewidth=3)
    plt.xlabel(xlbl, fontsize=14)
    plt.ylabel(ylbl, fontsize=14)
    plt.title('Accuracy is %.2f ' % acc)
    plt.legend(loc="upper left", fontsize=14)
    plt.axis(axes_lim)
    plt.show()

In [ ]:
plot_2D_classifier(X_test_2feat, y_test, log_reg, boundary, acc, xlbl=xlbl, ylbl=ylbl)

## C9: 3D Classification (three features)

Now select 3 features. Use `x1` and `x2` **as given** to calculate the boundary decision (a plane).

In [ ]:
#C9
#----------------------Implement your code here:------------------------------

# Select 3 features: radius_worst, compactness_worst, concavity_worst
X_train_3feat_arr = X_train[['radius_worst', 'compactness_worst', 'concavity_worst']].values
x_test = X_test[['radius_worst', 'compactness_worst', 'concavity_worst']].values

# Fit logistic regression
log_reg = LogisticRegression(random_state=10, max_iter=10000)
log_reg.fit(X_train_3feat_arr, y_train)

# Calculate accuracy
acc = log_reg.score(x_test, y_test)
print('Accuracy with 3 features: %.2f%%' % (100 * acc))

# Calculate decision boundary plane
# The decision boundary: w0*x0 + w1*x1 + w2*x2 + b = 0
# => x2 = -(w0*x0 + w1*x1 + b) / w2
w = log_reg.coef_[0]
b = log_reg.intercept_[0]

# Create meshgrid for the plane
x1 = np.linspace(x_test[:, 0].min(), x_test[:, 0].max(), 50)
x2 = np.linspace(x_test[:, 1].min(), x_test[:, 1].max(), 50)
x1, x2 = np.meshgrid(x1, x2)

# Calculate boundary surface: z = -(w0*x + w1*y + b) / w2
boundary = -(w[0] * x1 + w[1] * x2 + b) / w[2]

#------------------------------------------------------------------------------

In [ ]:
def plot_3D_classifier(x_test, y_test, x1, x2, boundary, acc, xlbl="radius_worst [mm]", ylbl="compactness_worst [N.U]", zlbl="concavity_worst [N.U]",
                       el=42, az=215):

    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection='3d')

    ax.scatter(x_test[y_test==0, 0], x_test[y_test==0, 1], x_test[y_test==0, 2], c=x_test[y_test==0, 2], cmap='Blues_r', label='B');
    ax.scatter(x_test[y_test==1, 0], x_test[y_test==1, 1], x_test[y_test==1, 2], c=x_test[y_test==1, 2], cmap='Reds_r', label='M');
    ax.plot_surface(x1, x2, boundary, cmap=cm.gray,
                            linewidth=0, antialiased=False, alpha=0.5)
    ax.view_init(el, az)
    ax.set_xlabel(xlbl, fontsize=14)
    ax.set_ylabel(ylbl, fontsize=14)
    ax.set_zlabel(zlbl, fontsize=14)
    plt.legend(loc="best", fontsize=14)
    plt.title('Accuracy is %.2f ' % acc)
    plt.show()

In [ ]:
plot_3D_classifier(x_test, y_test, x1, x2, boundary, acc)

---
*Notice the change in performance.*

---

## Usage in healthcare and conclusions

#### In this tutorial we:
>- implemented a classifier to diagnose breast lumps.

>- demonstrated how to implement logistic regression using `scikit-learn` built-in function.

>- saw the impact of standardization on classification's performances.

>- saw the impact of *feature selection*.

>- used 1D, 2D and 3D figures to illustrate the separation between binary labels.

#### Even though we deal with medical topics, there are still cases where the simplest models perform very well. Keep it in mind for the future.